# 02 - Preprocesamiento / Features
Limpieza de datos, feature engineering y preparación de matrices para modelado.


1. Carga y limpieza básica.
2. Análisis de faltantes para descartar columnas (>20% nulos) e imputar el resto.
3. Ingeniería de características (diferencias, promedios, amplitudes, rolling, etc.).
4. Codificación:
   - Ciclo (sin/cos) para direcciones de viento (WindDir9am, WindDir3pm, WindGustDir).
   - Agrupación de categorías raras en Location y One-Hot.
5. Construcción de pipeline de preprocesamiento (imputación selectiva + escalado + encoding).
6. Selección de variables por alta correlación (umbral 0.85) comparando desempeño (ROC AUC) con vs sin reducción usando validación cruzada.
7. Evaluación de transformaciones (log1p) en variables muy sesgadas (Rainfall, Evaporation).
8. Estrategias de desbalance: Dummy vs LogisticRegression (class_weight) vs RandomForest y SMOTE.
9. Generación de DataFrames finales listos para modelado: X_train_final, X_valid_final, y_train, y_valid, feature_names.
10. Resumen final de decisiones.


In [ ]:
# Importaciones y funciones
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

TARGET_COL = 'RainTomorrow'
RANDOM_STATE = 42
na_values = ["NA", "", "?", "NaN", "nan"]

def load_weather_data(path):
    return pd.read_csv(path, na_values=na_values)

def initial_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().replace(' ', '_') for c in df.columns]
    before = len(df)
    df = df.drop_duplicates()
    if len(df)!=before:
        print('Filas duplicadas removidas:', before-len(df))
    df = df[~df[TARGET_COL].isna()]
    return df


In [25]:
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if set(['MaxTemp','MinTemp']).issubset(df.columns):
        df['TempRange'] = df['MaxTemp'] - df['MinTemp']
    if set(['Humidity9am','Humidity3pm']).issubset(df.columns):
        df['HumidityDiff'] = df['Humidity3pm'] - df['Humidity9am']
    if 'WindGustSpeed' in df.columns:
        df['HighWindGust'] = (df['WindGustSpeed'] > df['WindGustSpeed'].median()).astype(int)
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df['Month'] = df['Date'].dt.month
        # Rolling simple por estación (esto es aproximado)
        if 'Rainfall' in df.columns:
            df = df.sort_values('Date')
            df['Rainfall_rolling3'] = df['Rainfall'].rolling(3, min_periods=1).mean()
    return df


In [27]:
# Carga y limpieza
DATA_PATH = Path('../data/weatherAUS.csv')
df_raw = load_weather_data(DATA_PATH)

df_clean = initial_cleaning(df_raw)

df_feat = add_derived_features(df_clean)
print('Nuevas columnas añadidas. Total cols:', len(df_feat.columns))

Nuevas columnas añadidas. Total cols: 28


In [36]:
# Análisis de faltantes y descarte >20%
missing_pct = df_clean.isna().mean()*100
cols_high_missing = missing_pct[missing_pct>20].index.tolist()
print('Columnas >20% nulos:', cols_high_missing)

df_reduced = df_clean.drop(columns=cols_high_missing)
print('Shape tras descarte:', df_reduced.shape)

Columnas >20% nulos: ['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm']
Shape tras descarte: (142193, 19)


In [29]:
# Ingeniería de características derivadas específicas
fe = df_reduced.copy()
# Diferencias y amplitudes
if {'Temp3pm','Temp9am'}.issubset(fe.columns):
    fe['TempDiff_3pm_9am'] = fe['Temp3pm'] - fe['Temp9am']
if {'Pressure3pm','Pressure9am'}.issubset(fe.columns):
    fe['PressureDiff_3pm_9am'] = fe['Pressure3pm'] - fe['Pressure9am']
if {'MaxTemp','MinTemp'}.issubset(fe.columns):
    fe['DailyTempRange'] = fe['MaxTemp'] - fe['MinTemp']
if {'Humidity3pm','Humidity9am'}.issubset(fe.columns):
    fe['HumidityDiff_3pm_9am'] = fe['Humidity3pm'] - fe['Humidity9am']
# Promedios diarios
if {'Temp3pm','Temp9am'}.issubset(fe.columns):
    fe['TempMean_Day'] = fe[['Temp3pm','Temp9am']].mean(axis=1)
if {'Pressure3pm','Pressure9am'}.issubset(fe.columns):
    fe['PressureMean_Day'] = fe[['Pressure3pm','Pressure9am']].mean(axis=1)
# Rolling rainfall promedio corto 
if 'Date' in fe.columns and 'Rainfall' in fe.columns:
    fe['Date'] = pd.to_datetime(fe['Date'], errors='coerce')
    fe = fe.sort_values('Date')
    fe['Rainfall_roll3'] = fe['Rainfall'].rolling(3, min_periods=1).mean()
print('Nuevas columnas derivadas añadidas. Total:', len(fe.columns))


Nuevas columnas derivadas añadidas. Total: 26


In [30]:
# Codificación cíclica y agrupación de categorías raras
work = fe.copy()
wind_cols = [c for c in work.columns if c.lower().startswith('winddir')]
wind_dirs = ['N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSW','SW','WSW','W','WNW','NW','NNW']
angle_map = {d: i * 360/len(wind_dirs) for i,d in enumerate(wind_dirs)}
for col in wind_cols:
    ang = work[col].map(angle_map)
    rad = np.deg2rad(ang)
    work[col+'_sin'] = np.sin(rad)
    work[col+'_cos'] = np.cos(rad)
# Rare category grouping para Location
if 'Location' in work.columns:
    freq = work['Location'].value_counts(normalize=True)
    rare_threshold = 0.01
    rares = freq[freq < rare_threshold].index
    work['Location_grouped'] = work['Location'].where(~work['Location'].isin(rares), other='__OTHER__')
print('Codificación cíclica aplicada y rare grouping listo.')


Codificación cíclica aplicada y rare grouping listo.


In [31]:
# Pipeline de preprocesamiento
from sklearn.preprocessing import RobustScaler
from sklearn.base import BaseEstimator, TransformerMixin

proc_df = work.copy()

target_series = proc_df[TARGET_COL].apply(lambda x: 1 if str(x).strip().lower()=='yes' else 0)
proc_df = proc_df.drop(columns=[TARGET_COL])

num_cols_all = proc_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_all = proc_df.select_dtypes(exclude=[np.number]).columns.tolist()

# Variables muy sesgadas para posible log1p (si >=0)
skew_candidates = ['Rainfall','Evaporation']
apply_log_cols = [c for c in skew_candidates if c in proc_df.columns]
for c in apply_log_cols:
    if (proc_df[c] >= 0).all():
        proc_df[c+'_log1p'] = np.log1p(proc_df[c])

In [32]:
# Dividir train/valid
from sklearn.model_selection import train_test_split
X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(proc_df, target_series, test_size=0.2, stratify=target_series, random_state=RANDOM_STATE)

num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()


In [33]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Imputación simple (<20% nulos); ya descartamos >20%
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# Escalado: Robust para columnas log1p originales potencialmente con outliers, Standard para el resto
robust_targets = [c for c in num_cols if any(base in c for base in ['Rainfall','Evaporation'])]
standard_targets = [c for c in num_cols if c not in robust_targets]

class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns): self.columns=columns
    def fit(self, X, y=None): return self
    def transform(self, X): return X[self.columns]

numeric_transformers = []
if standard_targets:
    numeric_transformers.append(('std', Pipeline([('sel', ColumnSelector(standard_targets)), ('imp', num_imputer), ('sc', StandardScaler())]), standard_targets))
if robust_targets:
    numeric_transformers.append(('rob', Pipeline([('sel', ColumnSelector(robust_targets)), ('imp', num_imputer), ('sc', RobustScaler())]), robust_targets))

cat_pipeline = Pipeline([
    ('imp', cat_imputer),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor_adv = ColumnTransformer(numeric_transformers + [('cat', cat_pipeline, cat_cols)])

X_train_proc = preprocessor_adv.fit_transform(X_train_raw)
X_valid_proc = preprocessor_adv.transform(X_valid_raw)

print('Shapes avanzado:', X_train_proc.shape, X_valid_proc.shape)


Shapes avanzado: (113754, 3575) (28439, 3575)


In [13]:
# Selección de variables por alta correlación (>=0.85) y comparación
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Construir dataframe numérico imputado temporal para medir correlaciones
X_train_num = X_train_raw[num_cols].copy()
X_train_num_imputed = pd.DataFrame(num_imputer.fit_transform(X_train_num), columns=num_cols, index=X_train_num.index)

corr_matrix = X_train_num_imputed.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [column for column in upper.columns if any(upper[column] >= 0.85)]
print('Columnas candidatas a remover por alta correlación:', high_corr_cols)

# Modelo baseline logistic con y sin remover
logit = LogisticRegression(max_iter=500, class_weight='balanced')
cv_scores_full = cross_val_score(logit, preprocessor_adv.fit_transform(X_train_raw), y_train, cv=3, scoring='roc_auc')
print('ROC AUC full mean:', cv_scores_full.mean())

# Remover columnas correlacionadas del raw y reconstruir pipeline
X_train_reduced = X_train_raw.drop(columns=high_corr_cols, errors='ignore')
num_cols_red = X_train_reduced.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_red = X_train_reduced.select_dtypes(exclude=[np.number]).columns.tolist()
preprocessor_red = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols_red),
    ('cat', cat_pipeline, cat_cols_red)
])
cv_scores_red = cross_val_score(logit, preprocessor_red.fit_transform(X_train_reduced), y_train, cv=3, scoring='roc_auc')
print('ROC AUC reduced mean:', cv_scores_red.mean())

if cv_scores_red.mean() >= cv_scores_full.mean() - 0.002:
    cols_selected_strategy = 'reduced'
    selected_columns = [c for c in X_train_reduced.columns]
    print('Se adopta versión reducida (no pérdida significativa).')
else:
    cols_selected_strategy = 'full'
    selected_columns = [c for c in X_train_raw.columns]
    print('Se mantiene versión completa.')


Columnas candidatas a remover por alta correlación: ['Pressure3pm', 'Temp9am', 'Temp3pm', 'TempMean_Day', 'PressureMean_Day']
ROC AUC full mean: 0.8751096720788732
ROC AUC reduced mean: 0.8751616447932897
Se adopta versión reducida (no pérdida significativa).


In [37]:
# Evaluación transformaciones (log1p) en variables sesgadas
from sklearn.metrics import roc_auc_score

skew_vars = [v for v in ['Rainfall','Evaporation'] if v in X_train_raw.columns]
print('Vars sesgadas evaluadas:', skew_vars)

# submuestreo para acelerar (versión completa era muy demorada)
FRACTION = 0.35  # 35% del train 
if len(X_train_raw) > 60000:
    sample_idx = X_train_raw.sample(frac=FRACTION, random_state=RANDOM_STATE).index
    X_train_eval = X_train_raw.loc[sample_idx].copy()
    y_train_eval = y_train.loc[sample_idx]
else:
    X_train_eval = X_train_raw.copy()
    y_train_eval = y_train.copy()

results_transform = []
for use_transform in [False, True]:
    X_tmp = X_train_eval.copy()
    if use_transform:
        for v in skew_vars:
            if (X_tmp[v] >= 0).all():
                X_tmp[v + '_log1p_alt'] = np.log1p(X_tmp[v])
    num_cols_tmp = X_tmp.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols_tmp = X_tmp.select_dtypes(exclude=[np.number]).columns.tolist()

    prep_tmp = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols_tmp),
        ('cat', cat_pipeline, cat_cols_tmp)
    ])

    X_train_enc = prep_tmp.fit_transform(X_tmp)
    # Transformar valid set de forma consistente
    X_valid_tmp = X_valid_raw.copy()
    if use_transform:
        for v in skew_vars:
            if (X_valid_tmp[v] >= 0).all():
                X_valid_tmp[v + '_log1p_alt'] = np.log1p(X_valid_tmp[v])
    X_valid_enc = prep_tmp.transform(X_valid_tmp)

    model = LogisticRegression(max_iter=300, class_weight='balanced', n_jobs=None)
    model.fit(X_train_enc, y_train_eval)
    proba = model.predict_proba(X_valid_enc)[:, 1]
    auc_val = roc_auc_score(y_valid, proba)
    results_transform.append({'use_transform': use_transform, 'logit_auc': auc_val})
    print(f"Transform={use_transform} AUC={auc_val:.4f}")

chosen_transform = max(results_transform, key=lambda d: d['logit_auc'])
print('Estrategia seleccionada (transformaciones):', chosen_transform)

# Bandera para pasos posteriores
USE_LOG_TRANSFORM = chosen_transform['use_transform']

Vars sesgadas evaluadas: ['Rainfall']
Transform=False AUC=0.8731
Transform=True AUC=0.8731
Estrategia seleccionada (transformaciones): {'use_transform': False, 'logit_auc': 0.8731418049453284}


In [21]:
# Estrategias de desbalance y dataset final
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# Seleccionar conjunto de columnas final según estrategia correlación
y_cols =  selected_columns
X_train_sel = X_train_raw[y_cols].copy()
X_valid_sel = X_valid_raw[y_cols].copy()

# Reconstruir preprocessor simple para finales
num_cols_final = X_train_sel.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_final = X_train_sel.select_dtypes(exclude=[np.number]).columns.tolist()
final_preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols_final),
    ('cat', cat_pipeline, cat_cols_final)
])
X_train_enc = final_preprocessor.fit_transform(X_train_sel)
X_valid_enc = final_preprocessor.transform(X_valid_sel)

models = {}
logit = LogisticRegression(max_iter=400, class_weight='balanced')
logit.fit(X_train_enc, y_train)
proba_logit = logit.predict_proba(X_valid_enc)[:,1]
models['logit_class_weight_auc'] = roc_auc_score(y_valid, proba_logit)

rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced_subsample')
rf.fit(X_train_enc, y_train)
proba_rf = rf.predict_proba(X_valid_enc)[:,1]
models['rf_balanced_auc'] = roc_auc_score(y_valid, proba_rf)

sm = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.5)
X_res, y_res = sm.fit_resample(X_train_enc, y_train)
rf_sm = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf_sm.fit(X_res, y_res)
proba_rf_sm = rf_sm.predict_proba(X_valid_enc)[:,1]
models['rf_smote_auc'] = roc_auc_score(y_valid, proba_rf_sm)

print('AUC modelos:', models)

# Guardar estructuras finales
X_train_final = X_train_enc
X_valid_final = X_valid_enc
feature_names = (list(final_preprocessor.named_transformers_['num'].get_feature_names_out(num_cols_final)) +
                 list(final_preprocessor.named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(cat_cols_final)))
print('Total features finales:', len(feature_names))


AUC modelos: {'logit_class_weight_auc': 0.8814845227431074, 'rf_balanced_auc': 0.8891942122026476, 'rf_smote_auc': 0.888100733694493}
Total features finales: 3570


In [43]:
# Exportar dataset preparado y metadatos para Notebook 03
import json
from pathlib import Path

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Dataset base con columnas seleccionadas (antes de encoding) + target
# Usamos X_train_raw y X_valid_raw para reconstruir conjunto completo original (train+valid)
combined_raw = pd.concat([X_train_raw[selected_columns], X_valid_raw[selected_columns]], axis=0)
combined_target = pd.concat([y_train, y_valid], axis=0)
export_df = combined_raw.copy()
export_df[TARGET_COL] = combined_target

csv_path = processed_dir / 'weather_prepared_selected.csv'
export_df.to_csv(csv_path, index=False)

## Conclusiones Preprocesamiento y FE

### Objetivo
Dejar definidas las decisiones de preprocesamiento, selección de variables y estrategias frente al desbalance para usar directamente en el notebook 03_modeling.

### 1. Datos y Particiones
- Variable objetivo: `RainTomorrow` (binaria: Yes→1, No→0).
- Split estratificado 80/20 (train/valid) con `random_state = 42`.
- Filas con objetivo nulo eliminadas; duplicados removidos si existían.

### 2. Limpieza y Faltantes
- Columnas con >20% nulos eliminadas: `Evaporation`, `Sunshine`, `Cloud9am`, `Cloud3pm`.
- Imputación numérica: Mediana.
- Imputación categórica: Moda (frecuente).

### 3. Ingeniería de Características
Incluye (si columnas base estaban presentes):
- Rangos y diferencias: `TempRange`, `TempDiff_3pm_9am`, `PressureDiff_3pm_9am`, `DailyTempRange`, `HumidityDiff`, `HumidityDiff_3pm_9am`.
- Promedios diarios: `TempMean_Day`, `PressureMean_Day`.
- Rolling / suavizado: `Rainfall_roll3`.
- Codificación cíclica para direcciones de viento: columnas `WindDir*` → pares `_sin` y `_cos`.
- Agrupación de categorías raras de `Location` (<1%) en `__OTHER__` → nueva `Location_grouped`.
- Transformaciones log potenciales: se agrega `Rainfall_log1p`.

### 4. Transformaciones y Escalado
- Estrategia combinada: escalado estándar para numéricas generales y robust para variables susceptibles a outliers (`Rainfall`).
- One-Hot Encoding para categóricas (incluye `Location_grouped`). `handle_unknown='ignore'` para robustez.
- Evaluación de añadir columnas log1p mostró el mismo AUC con o sin transformación.
  - Por tanto, se prefiere continuar sin transformación.

### 5. Selección de Variables por Correlación
- Matriz de correlación sobre numéricas imputadas.
- Umbral: |r| ≥ 0.85 para marcar candidatas a eliminar.
- Comparación Logistic Regression (AUC CV=3) full vs reduced.
- Decisión adoptada: se descartan variables `Pressure3pm`, `Temp9am`, `Temp3pm`, `TempMean_Day`, `PressureMean_Day` por no tener aporte en el modelo.

### 6. Dataset Final Para Modelado
- Subconjunto aplicado: `X_train_sel`, `X_valid_sel` sobre `selected_columns`.
- Preprocesador final (`final_preprocessor`): `SimpleImputer(median)` + `StandardScaler` para numéricas + `cat_pipeline` (imputación frecuencia + OHE) para categóricas.
- Matrices finales: `X_train_final`, `X_valid_final` (sparse matrices) y lista de `feature_names` ya alineada.

### 7. Desbalance de Clases
- Modelos probados:
  - LogisticRegression con `class_weight='balanced'` → AUC: `models['logit_class_weight_auc']`.
  - RandomForest con `class_weight='balanced_subsample'` → AUC: `models['rf_balanced_auc']`.
  - RandomForest tras SMOTE (ratio 0.5) → AUC: `models['rf_smote_auc']`.
- Conclusión preliminar:
  - Conservar ambas aproximaciones (ponderación de clases y SMOTE) para validación cruzada más rigurosa en evaluación de modelos.
  - Si diferencia de AUC de SMOTE vs class_weight es marginal (<~0.005), preferir solo class_weight (menor riesgo de overfitting y menor costo computacional). Revisar valores impresos reales antes de decidir.


### Observaciones
- Eliminación >20% nulos reduce varianza de imputación y evita sesgo de columnas muy incompletas.
- Correlación alta removida solo si no degradó AUC (tolerancia 0.002) para mantener parsimonia.
- Codificación cíclica preserva continuidad angular evitando discontinuidad N→NNW.
- Uso de `class_weight` prioritario por simplicidad; SMOTE evaluado para mejorar recall minoritaria si evidencia lo respalda.
